# Primary BEMD ablation on Kaggle (50 epochs)

Enable a GPU and Internet in Kaggle notebook settings, and attach the private raw MRI and BEMD cache datasets. Run the cells in order. The training cell starts the five primary conditions; earlier cells only set up and validate. All fixed experimental controls come from the existing repository config. No BEMD decomposition is recomputed.

Kaggle notebook settings: https://www.kaggle.com/docs/notebooks


In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir(), "Run this notebook on Kaggle."

REPO = WORK / "cardiac-mri-4ch-segmentation"
COMMIT = "b854779f727c83dbceafa42dce77bce702db5e5d"
URL = "https://github.com/sancia-jw/cardiac-mri-4ch-segmentation.git"

if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
assert head == COMMIT, f"Existing checkout is {head}; expected {COMMIT}."
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")],
    check=True,
)
print("Repository and dependencies ready:", head)


Locate inputs. Leave the overrides as None when only one matching dataset is attached. If there are multiple matches, supply the actual paths. DATA must contain 4CH_TR/image and 4CH_TR/anno; CACHE must contain the case folders and manifest directly.

In [ ]:
RAW_OVERRIDE = None    # Example: "/kaggle/input/cmr-multi-4ch/CINE_MULTI"
CACHE_OVERRIDE = None  # Example: "/kaggle/input/cmr-multi-bemd-square-pad"

def choose_one(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise RuntimeError(
            f"Expected one {label}; found {len(paths)}: "
            f"{[str(p) for p in paths]}. Attach the extracted dataset "
            "or set the corresponding override above."
        )
    return paths[0]

DATA = Path(RAW_OVERRIDE) if RAW_OVERRIDE else choose_one(
    [p.parent for p in INPUT.rglob("4CH_TR")
     if (p / "image").is_dir() and (p / "anno").is_dir()],
    "raw MRI root",
)
CACHE = Path(CACHE_OVERRIDE) if CACHE_OVERRIDE else choose_one(
    [p.parent for p in INPUT.rglob("cache_manifest.json")
     if (p.parent / "CINE_4CH_001").is_dir()],
    "BEMD cache root",
)
OUT = WORK / "bemd_ablation_primary"
LOGS = WORK / "bemd_primary_logs"
EXCLUSIONS = CACHE / "exclusions.json"
CONFIG = REPO / "configs/bemd_ablation.yaml"
SPLITS = REPO / "outputs/splits_4ch.csv"

assert (DATA / "4CH_TR/image").is_dir()
assert (DATA / "4CH_TR/anno").is_dir()
assert EXCLUSIONS.is_file(), f"Missing exclusions: {EXCLUSIONS}"
assert SPLITS.is_file(), f"Missing fixed split: {SPLITS}"

for name, path in [("DATA", DATA), ("CACHE", CACHE), ("OUT", OUT)]:
    print(f"{name}: {path}")


Check the GPU, primary conditions, and uploaded manifest. Set the epoch count and which conditions to train here. This does not construct or train a model.

In [ ]:
import hashlib
import json
import torch
import yaml

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

PRIMARY = [
    "original",
    "subtract_bimf_0",
    "subtract_bimf_1",
    "subtract_bimf_2",
    "subtract_bimf_0_1",
]
cfg = yaml.safe_load(CONFIG.read_text())
assert cfg["train"]["runs"] == PRIMARY
assert (
    cfg["train"]["epochs"], cfg["train"]["batch_size"],
    cfg["train"]["lr"], cfg["train"]["seed"],
) == (15, 4, 0.001, 42)

manifest = json.loads((CACHE / "cache_manifest.json").read_text())
exclusions = json.loads(EXCLUSIONS.read_text())
assert manifest["decomposition_method"] == "bemd_default_square_pad"
assert manifest["total_frames"] == 8285
assert manifest["valid_usable_frames"] == 8282
assert manifest["usable_bimf_availability"] == {
    "0": 8282, "1": 8282, "2": 8282, "3": 7385,
}
assert hashlib.sha256(EXCLUSIONS.read_bytes()).hexdigest() == manifest["exclusions_sha256"]
assert {(r["case_stem"], r["frame_idx"]) for r in exclusions["excluded_frames"]} == {
    ("CINE_4CH_023", 16), ("CINE_4CH_051", 65), ("CINE_4CH_085", 65),
}
# Overrides the config's 15 epochs via --epochs; everything else stays fixed.
EPOCHS = 50
# To split across Kaggle sessions, e.g. PRIMARY[:3] in one run and PRIMARY[3:] in another.
RUNS = PRIMARY
assert RUNS and set(RUNS) <= set(PRIMARY)
print("Conditions this run:", RUNS)
print("Epochs:", EPOCHS)
print("Usable frames: 8,282; exclusions: 3.")


Validate the mounted inputs using the actual CLI. This checks all cases and cache headers without training, evaluation, or decomposition. Console output is also saved to a log.

In [ ]:
LOGS.mkdir(parents=True, exist_ok=True)

BASE = [
    sys.executable, "-u", str(REPO / "scripts/run_bemd_ablation.py"),
    "--config", str(CONFIG),
    "--data-root", str(DATA),
    "--cache-root", str(CACHE),
    "--exclusions", str(EXCLUSIONS),
    "--splits-csv", str(SPLITS),
    "--output-root", str(OUT),
    "--device", "cuda",
    "--epochs", str(EPOCHS),
    "--runs", *RUNS,
]

def run_logged(command, log_name):
    log_path = LOGS / log_name
    print("Running:", " ".join(command))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            command, cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f"Command exited with code {code}; see {log_path}")

VALIDATION_PASSED = False
run_logged(BASE + ["--validate-only"], "validation.log")
VALIDATION_PASSED = True
print("Validation passed. No model has been trained.")


START TRAINING: this cell runs the conditions in RUNS for EPOCHS epochs and their existing test evaluation on Kaggle. It uses the fixed configuration and exclusions. It refuses to overwrite existing run artifacts. For a separate fresh experiment, change OUT before validation; partial-training recovery is not automatically applied.

In [ ]:
assert INPUT.is_dir() and WORK.is_dir(), "Training is intended for Kaggle."
assert VALIDATION_PASSED, "Run the validation cell successfully first."
assert torch.cuda.is_available(), "A GPU is required."

existing = [OUT / run_id for run_id in RUNS if (OUT / run_id).exists()]
if existing:
    raise RuntimeError(
        "Existing run artifacts found; refusing to overwrite them: "
        + ", ".join(str(p) for p in existing)
    )

run_logged(BASE, "training.log")
print(f"{len(RUNS)} condition(s) x {EPOCHS} epochs and test evaluations completed.")


Review and download results. Save a Kaggle notebook version with its outputs after the run. The summary, metrics, checkpoints, and derived image caches are under /kaggle/working/bemd_ablation_primary; console logs are under /kaggle/working/bemd_primary_logs. Input datasets remain untouched.

In [ ]:
import pandas as pd
from IPython.display import display, FileLink

summary_path = OUT / "ablation_summary.csv"
assert summary_path.is_file(), "The combined summary is not available yet."
summary = pd.read_csv(summary_path)
assert len(summary) == len(RUNS) and set(summary["run_id"]) == set(RUNS)

display(summary[[
    "run_id", "best_epoch",
    "val_dice_mean_foreground", "test_dice_mean_foreground",
    "delta_test_fg_vs_original",
]])

display(FileLink(str(summary_path)))
display(FileLink(str(OUT / "ablation_summary.json")))
display(FileLink(str(LOGS / "training.log")))
for run_id in RUNS:
    checkpoint = OUT / run_id / "checkpoints/best.pt"
    assert checkpoint.is_file(), f"Missing best checkpoint: {checkpoint}"
    display(FileLink(str(checkpoint)))
